In [12]:
import pandas as pd
import numpy as np
import xgboost as xgb
import os

from sklearn.preprocessing import LabelEncoder

In [4]:
# CSV Master (path to be adjusted accordingly)
file_path = "../raw_data/respiratory_features_master_forXGboost.csv"

if os.path.exists(file_path):
    df = pd.read_csv(file_path)
    print(f"done! {len(df)} lines")
    
    # check NaNs
    nans = df.isnull().sum().sum()
    print(f"missing values: {nans}")
    
    # columns
    print(f"cols: {df.columns.tolist()}")
    display(df.head())
else:
    print("check path")

done! 920 lines
missing values: 0
cols: ['rms_mean', 'zcr_mean', 'centroid_mean', 'rolloff_mean', 'flux_mean', 'mfcc_0', 'mfcc_1', 'mfcc_2', 'mfcc_3', 'mfcc_4', 'mfcc_5', 'mfcc_6', 'mfcc_7', 'mfcc_8', 'mfcc_9', 'mfcc_10', 'mfcc_11', 'mfcc_12', 'filename', 'patient_id']


,rms_mean,zcr_mean,centroid_mean,rolloff_mean,flux_mean,mfcc_0,mfcc_1,mfcc_2,mfcc_3,mfcc_4,mfcc_5,mfcc_6,mfcc_7,mfcc_8,mfcc_9,mfcc_10,mfcc_11,mfcc_12,filename,patient_id
0,0.353043,0.000716,43.795286,41.528320,0.433962,-416.897430,62.325581,46.420395,35.231888,30.405567,29.885906,28.196938,24.402584,21.075634,19.104095,17.423349,14.515306,10.367791,223_1b1_Pr_sc_Meditron.wav,223
1,0.131263,0.001339,133.438929,65.929846,0.431228,-488.044525,75.769440,61.811069,49.042362,39.135307,31.911039,25.999527,21.384661,17.560677,14.248250,11.581615,10.005250,9.039628,134_2b2_Al_mc_LittC2SE.wav,134
2,0.147832,0.023517,1299.190164,2820.226061,1.032007,-217.973022,100.233185,4.600662,29.796114,15.727752,17.164007,14.577301,18.850311,10.304487,13.588282,5.419698,6.978378,4.364567,170_1b4_Al_mc_AKGC417L.wav,170
3,0.236638,0.007697,337.687454,289.700565,0.922199,-330.460999,147.148163,79.223549,52.871487,31.837748,21.586948,15.103443,11.056579,8.015654,6.217966,4.712320,3.982452,3.184886,203_1p3_Pl_mc_AKGC417L.wav,203
4,0.080208,0.026243,1100.981859,2400.411740,1.046043,-312.236053,141.208221,8.134517,29.663057,17.224943,30.943031,15.562644,-0.528191,-3.173283,9.954062,7.148951,6.205405,1.809195,207_2b3_Tc_mc_AKGC417L.wav,207


In [7]:
# load diagnosis 
diag_path = "../raw_data/patient_diagnosis.csv" 

df_diag = pd.read_csv(diag_path)

df_diag.head()

,101,URTI
0,102,Healthy
1,103,Asthma
2,104,COPD
3,105,URTI
4,106,COPD


In [9]:
df_diag = pd.read_csv(diag_path, names=['patient_id', 'diagnosis'])

df_diag

,patient_id,diagnosis
0,101,URTI
1,102,Healthy
2,103,Asthma
3,104,COPD
4,105,URTI
...,...,...
121,222,COPD
122,223,COPD
123,224,Healthy
124,225,Healthy


In [10]:
# int
df['patient_id'] = df['patient_id'].astype(int)
df_diag['patient_id'] = df_diag['patient_id'].astype(int)

# merge
df_final = pd.merge(df, df_diag, on='patient_id', how='left')

print(f"merge done")
print(f"shape: {df_final.shape}")
print(f"disease dist:\n{df_final['diagnosis'].value_counts()}")

display(df_final.head())

merge done
shape: (920, 21)
disease dist:
diagnosis
COPD              793
Pneumonia          37
Healthy            35
URTI               23
Bronchiectasis     16
Bronchiolitis      13
LRTI                2
Asthma              1
Name: count, dtype: int64


,rms_mean,zcr_mean,centroid_mean,rolloff_mean,flux_mean,mfcc_0,mfcc_1,mfcc_2,mfcc_3,mfcc_4,...,mfcc_6,mfcc_7,mfcc_8,mfcc_9,mfcc_10,mfcc_11,mfcc_12,filename,patient_id,diagnosis
0,0.353043,0.000716,43.795286,41.528320,0.433962,-416.897430,62.325581,46.420395,35.231888,30.405567,...,28.196938,24.402584,21.075634,19.104095,17.423349,14.515306,10.367791,223_1b1_Pr_sc_Meditron.wav,223,COPD
1,0.131263,0.001339,133.438929,65.929846,0.431228,-488.044525,75.769440,61.811069,49.042362,39.135307,...,25.999527,21.384661,17.560677,14.248250,11.581615,10.005250,9.039628,134_2b2_Al_mc_LittC2SE.wav,134,COPD
2,0.147832,0.023517,1299.190164,2820.226061,1.032007,-217.973022,100.233185,4.600662,29.796114,15.727752,...,14.577301,18.850311,10.304487,13.588282,5.419698,6.978378,4.364567,170_1b4_Al_mc_AKGC417L.wav,170,COPD
3,0.236638,0.007697,337.687454,289.700565,0.922199,-330.460999,147.148163,79.223549,52.871487,31.837748,...,15.103443,11.056579,8.015654,6.217966,4.712320,3.982452,3.184886,203_1p3_Pl_mc_AKGC417L.wav,203,COPD
4,0.080208,0.026243,1100.981859,2400.411740,1.046043,-312.236053,141.208221,8.134517,29.663057,17.224943,...,15.562644,-0.528191,-3.173283,9.954062,7.148951,6.205405,1.809195,207_2b3_Tc_mc_AKGC417L.wav,207,COPD


In [13]:
#crazy class imbalance. make XGboost ready:

In [14]:
# transform words (COPD, Healthy...) in num (0, 1, 2...)
le = LabelEncoder()
df_final['target'] = le.fit_transform(df_final['diagnosis'])

# so we know who is who
mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print(f"class mapping: {mapping}")

# define X and y (target)
# remove columns that are not numbers for now?
X = df_final.drop(columns=['filename', 'patient_id', 'diagnosis', 'target'])
y = df_final['target']
groups = df_final['patient_id'] # split per id

print(f"features: {X.shape[1]} cols")

class mapping: {'Asthma': np.int64(0), 'Bronchiectasis': np.int64(1), 'Bronchiolitis': np.int64(2), 'COPD': np.int64(3), 'Healthy': np.int64(4), 'LRTI': np.int64(5), 'Pneumonia': np.int64(6), 'URTI': np.int64(7)}
features: 18 cols
